# Module 3.2: Persistent Linux NFS Client Mounts (`fstab` Automount) & Network Storage Benchmarking

## UniFi Storage & Enterprise Networking (Chapter 3) - Step 3: NAS Storage Pool & File Sharing

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand UniFi OS's exact NFS export convention (`/var/nfs/shared/<ShareName>`) and why specifying **`vers=3,nolock,_netdev,nofail,x-systemd.automount`** in `/etc/fstab` avoids both UniFi OS NFSv4 locking quirks and Linux boot hangs if the NAS is offline.
2. Install `nfs-common` on both `homelab-pi` (`10.0.0.200`) and `homelab-gpu` (`10.0.0.201`), inspect live exports with `showmount -e`, and configure persistent mounts at `/mnt/unas_backups` and `/mnt/unas_media`.
3. Verify reboot resilience and clean read/write permissions across both nodes.
4. Benchmark real sequential read/write throughput across your `USW-Enterprise-8-PoE` switch (`homelab-pi` 1 GbE vs. `homelab-gpu` 1 GbE / 2.5 GbE) and mount `MediaArchive` via SMB on your personal workstation.

## 3.2.1 Conceptual Background: Resilient Linux NFS Mounting Against UniFi OS

Mounting a network filesystem onto an always-on Linux server requires more care than mounting a local USB SSD (`Chapter 1, Module 3.2`). If you use naïve default NFS mount flags in `/etc/fstab`, two real-world failure modes will bite you:

### 1. Boot-Time Hang Protection (`_netdev,nofail,x-systemd.automount`)
Suppose the power blinks and `homelab-pi` boots in 18 seconds while the `UNAS 2` takes 75 seconds to spin up both 4 TB drives and start its NFS daemon.
- **Without resilient flags:** Linux's `systemd` blocks the boot sequence waiting for `10.0.0.202` for 90+ seconds (or drops into emergency maintenance mode, breaking SSH and Pi-hole DNS!).
- **With `_netdev,nofail,x-systemd.automount,x-systemd.idle-timeout=600`:**
  - `_netdev` tells systemd the mount requires an active network interface.
  - `nofail` guarantees the host finishes booting cleanly even if `UNAS 2` is powered off for maintenance.
  - `x-systemd.automount` mounts the share **on demand** the instant any script, container, or user accesses `/mnt/unas_backups`—automatically recovering after a NAS reboot with zero manual `mount -a` intervention!

### 2. Why `vers=3,nolock` on UniFi Drive (`UNAS 2`)?
UniFi OS exports shared drives at `<UNAS_IP>:/var/nfs/shared/<SharedDriveName>`. While modern Linux clients default to negotiating NFSv4 (`vers=4.2`), Ubiquiti's UniFi Drive NFS implementation frequently exhibits file-locking (`rpc.statd` / UID mapping) stalls under NFSv4. Explicitly pinning **`vers=3,nolock`** (just as recommended in Ubiquiti's UniFi Drive community & engineering docs) yields rock-solid, low-overhead wire-speed transfers for backups and media streaming.

| Option Flag | Engineering Purpose |
| :--- | :--- |
| `vers=3` | Pins reliable NFSv3 protocol against UniFi OS's `/var/nfs/shared/*` exports |
| `nolock` | Disables legacy NLM lock daemon contention across multi-node Linux clients |
| `rw` / `ro` | Enforces read-write on `HomelabBackups` and read-only safety on Pi's `MediaArchive` |
| `relatime` | Eliminates unnecessary metadata write amplification on every file read |
| `_netdev,nofail` | Waits for network stack; never halts Linux boot if `UNAS 2` is offline |
| `x-systemd.automount` | Lazily mounts & auto-reconnects seamlessly whenever the directory is accessed |

## 3.2.2 Step-by-Step Laboratory Guide

### Task 3.2.1: Install `nfs-common` & Discover Exports on `homelab-pi` (`10.0.0.200`)
1. SSH into your Raspberry Pi (`ssh homelab-pi`), install the NFS client package, and query your `UNAS 2`'s live export table:
   ```bash
   sudo apt update && sudo apt install -y nfs-common
   showmount -e 10.0.0.202
   ```
   *(Expected output:)*
   ```text
   Export list for 10.0.0.202:
   /var/nfs/shared/HomelabBackups 10.0.0.200,10.0.0.201
   /var/nfs/shared/MediaArchive   10.0.0.200,10.0.0.201
   ```
   *(If `showmount` returns empty, double-check that NFS is toggled ON in UniFi Drive Settings $\rightarrow$ Services and `10.0.0.200` is listed under NFS client IPs!)*
2. Create the local mount point directories on `homelab-pi`:
   ```bash
   sudo mkdir -p /mnt/unas_backups /mnt/unas_media
   ```
3. Back up `/etc/fstab` and append the resilient NFS automount entries:
   ```bash
   sudo cp /etc/fstab /etc/fstab.bak.$(date +%F)

   cat << 'EOT' | sudo tee -a /etc/fstab

   # UniFi UNAS 2 (10.0.0.202) RAID 1 NFSv3 Persistent Automounts
   10.0.0.202:/var/nfs/shared/HomelabBackups  /mnt/unas_backups  nfs  rw,relatime,vers=3,nolock,_netdev,nofail,x-systemd.automount  0  0
   10.0.0.202:/var/nfs/shared/MediaArchive    /mnt/unas_media    nfs  ro,relatime,vers=3,nolock,_netdev,nofail,x-systemd.automount  0  0
   EOT
   ```
4. Reload `systemd`, activate the automount units, and verify capacity and write access:
   ```bash
   sudo systemctl daemon-reload
   sudo systemctl restart remote-fs.target
   ls -la /mnt/unas_backups /mnt/unas_media
   df -h /mnt/unas_backups /mnt/unas_media
   ```
   You should see **`~3.6T`** total size mounted over `10.0.0.202:/var/nfs/shared/...`!
5. Create structured subdirectories inside `/mnt/unas_backups` (confirming write permission works cleanly):
   ```bash
   mkdir -p /mnt/unas_backups/homelab-pi/{immich-db,immich-library,configs}
   mkdir -p /mnt/unas_backups/homelab-gpu/{unifi-network,configs}
   ls -la /mnt/unas_backups/homelab-pi
   ```

### Task 3.2.2: Configure Persistent NFS Mounts on `homelab-gpu` (`10.0.0.201`)
Now repeat the exact clean setup on your GPU & UniFi Controller node (`ssh homelab-gpu`), mounting both `HomelabBackups` and `MediaArchive` as Read/Write (so `homelab-gpu` can back up its UniFi Network configs and access `MediaArchive` for NVENC video processing or bulk transfers):

```bash
sudo apt update && sudo apt install -y nfs-common
showmount -e 10.0.0.202

sudo mkdir -p /mnt/unas_backups /mnt/unas_media
sudo cp /etc/fstab /etc/fstab.bak.$(date +%F)

cat << 'EOT' | sudo tee -a /etc/fstab

# UniFi UNAS 2 (10.0.0.202) RAID 1 NFSv3 Persistent Automounts
10.0.0.202:/var/nfs/shared/HomelabBackups  /mnt/unas_backups  nfs  rw,relatime,vers=3,nolock,_netdev,nofail,x-systemd.automount  0  0
10.0.0.202:/var/nfs/shared/MediaArchive    /mnt/unas_media    nfs  rw,relatime,vers=3,nolock,_netdev,nofail,x-systemd.automount  0  0
EOT

sudo systemctl daemon-reload
sudo systemctl restart remote-fs.target
df -h /mnt/unas_backups /mnt/unas_media
```

### 🖼️ Task 3.2.2 Verification: Active `UNAS 2` NFS Mounts on Both Nodes

**Verification Objective:** Capture a screenshot showing `df -h /mnt/unas_backups /mnt/unas_media` (and `mount | grep nfs`) succeeding on `homelab-pi` or `homelab-gpu` with ~3.6T available capacity from `10.0.0.202`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 3.2.3: Benchmark Sequential NFS Throughput & Test Workstation SMB Access
*(Wait until initial RAID 1 background sync reaches 100% in UniFi Drive before judging peak speed!)*

1. **Sequential Write & Read Benchmark (run on `homelab-gpu`, then optionally on `homelab-pi`):**
   ```bash
   # Write a 1 GB test file directly to the UNAS 2 RAID 1 array (bypassing local write cache with oflag=direct or conv=fdatasync)
   dd if=/dev/zero of=/mnt/unas_backups/bench_1gb.bin bs=4M count=256 conv=fdatasync status=progress

   # Read the 1 GB file back over the USW-Enterprise-8-PoE switch
   dd if=/mnt/unas_backups/bench_1gb.bin of=/dev/null bs=4M status=progress

   # Clean up the temporary benchmark file immediately
   rm -f /mnt/unas_backups/bench_1gb.bin
   ```
   - **Expected Results:**
     - Over a **1 GbE** host link (`homelab-pi` or built-in laptop NIC): **~105–116 MB/s** (~900 Mbps effective NFSv3 wire speed!).
     - Over a **2.5 GbE** host link (`homelab-gpu` with a USB-C `RTL8156` adapter): **~150–210 MB/s** sustained onto the 5400 RPM RAID 1 HDDs (and up to **~270 MB/s** on cached reads from UNAS 2's 4 GB RAM!).
2. **Workstation SMB Mount Check (macOS / Windows):**
   - **macOS Finder:** Press `Cmd + K` $\rightarrow$ enter `smb://10.0.0.202/MediaArchive` $\rightarrow$ sign in with your UniFi Drive user credentials.
   - **Windows Explorer:** Type `\\10.0.0.202\MediaArchive` in the address bar $\rightarrow$ sign in with your UniFi Drive user credentials.
   - Drag a test file into `MediaArchive` and verify it appears instantaneously inside `/mnt/unas_media` on `homelab-pi`!

### 🖼️ Task 3.2.3 Verification: Sequential NFS Benchmark Results

**Verification Objective:** Capture a screenshot showing your `dd` sequential write/read benchmark results to `/mnt/unas_backups` over your `USW-Enterprise-8-PoE` switch.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the final guide, confirm the following checklist:

- [ ] `nfs-common` installed on both `homelab-pi` and `homelab-gpu`, and `showmount -e 10.0.0.202` lists both shares.
- [ ] Resilient `/etc/fstab` automount entries (`vers=3,nolock,_netdev,nofail,x-systemd.automount`) active on both nodes.
- [ ] `HomelabBackups` directory structure created (`homelab-pi/` and `homelab-gpu/`).
- [ ] Sequential NFS throughput benchmarked cleanly across `USW-Enterprise-8-PoE`, and workstation SMB access to `MediaArchive` verified.
- [ ] Verification screenshots captured and attached above.

Proceed to [Module 4.1: University OneDrive Migration & Self-Hosted Web File Management (FileBrowser Quantum)](../04_file_management_and_onedrive_migration/04_1_onedrive_migration_and_filebrowser.ipynb).